# Capability 15: Temporal reasoning for current, historical, and comparative period analysis

8/8 cases passed against a real, live LLM (gateway-configured model, see `.env`). Every code cell below is real, executable code -- the same `ask()` pattern as `notebooks/demo.ipynb` -- not a mockup; the attached output is what actually happened when this ran, captured via `scripts/run_live_capability_tests.py --capability 15`. Re-running this notebook (Restart Kernel & Run All) with a live key will make new real calls.

See `tests/live/cases/cap15_temporal_reasoning.py` for these case definitions with their automated pass/fail checks, and `tests/live/live_capabilities_suite.py` for how they run as unittest assertions.

In [ ]:
import sys, pathlib

# Robust path insert regardless of where Jupyter's cwd lands (repo root, or
# this notebook's own folder under notebooks/capabilities/<slug>/):
_p = pathlib.Path.cwd()
while not (_p / "src").exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

import os

try:
    from dotenv import load_dotenv  # optional: picks up a .env file if python-dotenv is installed
    load_dotenv(override=False)
except ImportError:
    pass

from src.orchestrator import Orchestrator
from src.llm_client import get_llm_client, GLOBAL_USAGE, MockLLMClient

provider = os.environ.get("LLM_PROVIDER", "").lower() or ("anthropic" if os.environ.get("ANTHROPIC_API_KEY") else "openai" if os.environ.get("OPENAI_API_KEY") else "mock")
print(f"LLM provider in use: {provider}" + ("  (\u26a0\ufe0f set ANTHROPIC_API_KEY or OPENAI_API_KEY for real answers)" if provider == "mock" else ""))

orch = Orchestrator()


LLM provider in use: openai


In [ ]:
def ask(question: str, label: str = ""):
    """Run one turn through the orchestrator and pretty-print everything the
    grader needs to see: routing, evidence sources, transparency notes, answer.
    Identical helper to notebooks/demo.ipynb -- see scripts/build_notebook.py."""
    if label:
        print(f"\n{'='*90}\n{label}\n{'='*90}")
    print(f"USER: {question}\n")
    resp = orch.handle_turn(question)
    print(f"[intent={resp.intent} | sub_agents={resp.sub_agents_used} | retried={resp.retried}]")
    if resp.citations:
        print(f"[citations: {[c['doc_id'] for c in resp.citations]}]")
    if resp.assumptions:
        print("[assumptions/limitations surfaced:]")
        for a in resp.assumptions:
            print(f"  - {a}")
    print(f"\nAGENT: {resp.answer}")
    if resp.follow_up_suggestions:
        print(f"\n(follow-up suggestions: {resp.follow_up_suggestions})")
    return resp


## 01: current/latest period, implied not stated

**✅ PASS**

In [ ]:
_ = ask("What is AB InBev's most recent quarterly revenue?", '01')

01
USER: What is AB InBev's most recent quarterly revenue?

[intent=data_query | sub_agents=['structured']]

AGENT: AB InBev's most recent reported quarterly revenue for Global is **$15,384 million** in **Q4 2025** (period label: Q4 2025). [Source: structured KPI data]

If useful, I can check how that compares to the prior quarter or the same quarter a year earlier, or break it down by region/segment.

## 02: explicit historical period

**✅ PASS**

In [ ]:
_ = ask("What was North America's revenue in Q1 2024?", '02')

02
USER: What was North America's revenue in Q1 2024?

[intent=data_query | sub_agents=['structured']]

AGENT: North America's revenue in **Q1 2024** was **$3,593 million (USD)** [structured KPI data].

Want to compare this with Q1 2025 or break it down by brand/segment?

## 03: year-over-year comparison, explicit

**✅ PASS**

In [ ]:
_ = ask('How did Middle Americas revenue in Q4 2025 compare to Q4 2024?', '03')

03
USER: How did Middle Americas revenue in Q4 2025 compare to Q4 2024?

[intent=comparison | sub_agents=['structured']]

AGENT: **Middle Americas revenue — Q4 2025 vs. Q4 2024**

| Metric | Q4 2024 | Q4 2025 | Change |
| --- | --- | --- | --- |
| Revenue (USD m) | 4,395.0 | 4,927.0 | +532.0 (+12.1%) |

Revenue in Middle Americas grew by **USD 532.0m**, from **4,395.0m in Q4 2024 to 4,927.0m in Q4 2025** — an increase of roughly **12.1%** year over year.

Note: the retrieved evidence covers only total revenue for these two quarters; it does not include volume, price/mix, or currency-neutral breakdowns, so the drivers behind the increase can't be attributed from this data alone.

Follow-ups you might find useful:
- How did Middle Americas volume (or revenue per hectoliter) trend over the same two quarters?
- How does Middle Americas' Q4 2025 growth compare with other zones (e.g., North America or EMEA)?

## 04: quarter-over-quarter comparison

**✅ PASS**

In [ ]:
_ = ask("How did North America's revenue in Q2 2025 compare to Q1 2025?", '04')

04
USER: How did North America's revenue in Q2 2025 compare to Q1 2025?

[intent=comparison | sub_agents=['structured']]

AGENT: **North America revenue: Q2 2025 vs Q1 2025**

| Metric | Value |
| --- | --- |
| Q1 2025 revenue | $3,364.0M |
| Q2 2025 revenue | $3,844.0M |
| Change | +$480.0M |
| Change (%) | +14.27% |

North America's revenue rose sequentially in Q2 2025, up **$480.0M (+14.27%)** from Q1 2025. Keep in mind this is a quarter-over-quarter comparison, so part of the increase may reflect normal seasonality (Q2 covers the spring/summer selling season) rather than purely underlying growth.

*Evidence note: only the structured KPI query above was available for this question — no internal documents or web sources were retrieved, so I can't attribute the change to specific drivers such as volume, price/mix, or brand performance.*

Follow-ups you might find useful:
- How does Q2 2025 compare to Q2 2024 (year-over-year) for North America revenue?
- What drove the change — North A

## 05: full-year trend across all quarters

**✅ PASS**

In [ ]:
_ = ask("What is EMEA's revenue trend across each quarter of 2025?", '05')

05
USER: What is EMEA's revenue trend across each quarter of 2025?

[intent=data_query | sub_agents=['structured']]

AGENT: Based on the structured KPI data for EMEA in 2025, revenue trended upward through the year, then flattened in H2:

| Quarter | Revenue (USD million) |
| --- | --- |
| Q1 2025 | $1,965 |
| Q2 2025 | $2,489 |
| Q3 2025 | $2,524 |
| Q4 2025 | $2,524 |

**Trend summary:**
- **Q1 → Q2:** Strong growth of **+$524M**.
- **Q2 → Q3:** Modest growth of **+$35M**.
- **Q3 → Q4:** Flat, with revenue unchanged at **$2,524M**.

Overall, EMEA revenue grew from **$1,965M in Q1** to **$2,524M by Q4**, with most of the increase occurring in Q2.

If useful, I can pull the same quarterly trend for 2024 to compare year-over-year, or break down EMEA revenue by brand or subregion.

## 06: relative period phrasing ('last year', 'this year')

**✅ PASS**

In [ ]:
_ = ask("What was South America's revenue last year?", '06')

06
USER: What was South America's revenue last year?

[intent=data_query | sub_agents=['structured']]

AGENT: South America's revenue last year (2024) was **$12,423 million USD**.

This figure comes from the summed quarterly KPI data for the South America zone in 2024, as returned by the structured KPI query.

Note: your active context was set to 2025, so "last year" was interpreted as 2024. If you actually meant a different year, let me know and I can re-run the query.

Would you like to see how 2025 revenue compares so far, or a quarterly breakdown of the 2024 figure?

## 07: explicit multi-year comparison intent

**✅ PASS**

In [ ]:
_ = ask("Compare North America's EBITDA margin in 2024 versus 2025", '07')

07
USER: Compare North America's EBITDA margin in 2024 versus 2025

[intent=comparison | sub_agents=['structured']]

AGENT: **North America EBITDA margin: 2024 vs 2025**

| Year | EBITDA Margin |
| --- | --- |
| 2024 | 32.7% |
| 2025 | 33.0% |

**Takeaway:** North America's EBITDA margin expanded modestly year over year, rising **+0.3 percentage points** from 32.7% in 2024 to 33.0% in 2025 (roughly a 0.9% relative improvement). This is a marginal gain rather than a step change — profitability in the zone is holding broadly steady in the low-33% range.

Note on the data: these figures are computed from quarterly-grain records aggregated by year (sum of normalized EBITDA ÷ sum of revenue), so they reflect a full-year blend of quarterly performance rather than a single reported annual figure.

Follow-up ideas:
- How does North America's 2025 margin compare with the zone's trend over 2022–2023?
- Which other zones (e.g., South America, EMEA) had higher EBITDA margins in 2025?

## 08: annual (FY) grain versus quarterly grain distinction

**✅ PASS**

In [ ]:
_ = ask("What was AB InBev's FY2025 total revenue?", '08')

08
USER: What was AB InBev's FY2025 total revenue?

[intent=data_query | sub_agents=['structured']]

AGENT: AB InBev's FY2025 total global revenue was **$59,320 million USD** (approximately **$59.3 billion**), based on the structured KPI data for the Global zone at annual grain for FY2025.

Would you like a comparison with FY2024 revenue or a breakdown by region/segment?